In [1]:
import math
import matplotlib.pyplot as plt 
import random

In [ ]:
class Weight:
    def __init__(self, value, _children=(), _op="") :
        self.value = value
        self._prev = set(_children)
        self._backward = lambda: None
        self._op = _op
        self.grad = 0.0
        self.label = ""
        
    def __repr__(self): 
        return f"Weight(value={self.value:2f})"

    def __add__(self, other):
        out = Weight(self.value + other.value, (self,other), "+")
        def _backward():
            self.grad = out.grad 
            other.grad = out.grad

        out._backward = _backward
        return out

    def __mul__(self, other):
        out = Weight(self.value * other.value, (self,other), "*")
        def _backward():
            self.grad = other.value * out.grad
            other.grad = self.grad * out.grad

        out._backward = _backward
        return out 

    def ln(self):
        out = Weight(math.ln(self.value), _children=(self, ), _op='ln')
        def _backward():
            self.grad = self.value ** (-1) * out.grad
        out._backward = _backward
        return out

    def sigmoid(self):
        s = (1 + math.exp(-self.value))**(-1)
        out = Weight(value=s, _children=(self, ), _op='sigmoid')

        def _backward():
            self._grad = s * (1 - s) * out._grad

        out._backward = _backward
        return out

In [ ]:
class Matrix:
    def __init__(self, mat:list[list[Weight]]):
        self.matrix: list[list[Weight]] = mat
        self.rows = len(self.matrix)
        self.cols = len(self.matrix[0])
        self.size = self.rows, self.cols
        
    def __repr__(self):
        return "\n".join([" ".join([value.__repr__() for value in row]) for row in self.matrix])

    def __matmul__(self, other):
        if self.cols != other.rows:
            raise ValueError("Matrix size mismatch!")

        result = [[Weight(0) for _ in range(other.cols)] for _ in range(self.rows)]
        r = Matrix(result)
        
        A = self.matrix
        B = other.matrix

        # Triple nested loop
        for i in range(self.rows):               # rows of self
            for j in range(other.cols):         # columns of other
                for k in range(other.rows):    # rows of other (or columns of self)
                    result[i][j] = result[i][j] + A[i][k] * B[k][j]

        return Matrix(result)

    def __add__(self, other):
        if self.size != other.size:
            raise ValueError("Matrix size mismatch")

        result: list[list[Weight]] = [[None for _ in range(self.cols)] for _ in range(self.rows)]

        for i in range(self.rows):
            for j in range(self.cols):
                result[i][j] = self.matrix[i][j] + other.matrix[i][j]

        return Matrix(result)

    @staticmethod
    def wrap(matrix=list[list[float]]) -> list[list[Weight]]:
        for i in range(len(matrix)):
            for j in range(len(matrix[0])):
                matrix[i][j] = Weight(matrix[i][j])
        
        return matrix

class FullyConnectedLayer:
    def __init__(self, _in: int, _out: int, matrix=None, bias=None):
        if matrix is None:
            self.matrix = Matrix([[Weight(value=random.uniform(0,1)) for _ in range(_out)] for _ in range(_in)])
        else:
            self.matrix = Matrix(Matrix.wrap(matrix))

        if bias is None:
            self.bias = Matrix([[Weight(value=random.uniform(0,1)) for _ in range(_out)]])
        else:
            self.bias = Matrix(Matrix.wrap(bias))
        
    def __call__(self, x: Matrix):
        return x @ self.matrix + self.bias

class CrossEntropyLoss:
    def __init__(self):
        pass

    def __call__(self, y: Matrix, y_hat: Matrix):
        L = Weight(value=1/y.size[-1])
        

mlp = FullyConnectedLayer(2,4, matrix = [[1,1,1,1],[1,1,1,1]], bias=[[2,2,2,2]])

In [4]:
mlp.matrix, mlp.bias

(Weight(value=1.000000) Weight(value=1.000000) Weight(value=1.000000) Weight(value=1.000000)
 Weight(value=1.000000) Weight(value=1.000000) Weight(value=1.000000) Weight(value=1.000000),
 Weight(value=2.000000) Weight(value=2.000000) Weight(value=2.000000) Weight(value=2.000000))

In [5]:
x = Matrix([[Weight(1),Weight(2)]])
out = mlp(x)

res = Weight(0)
for v in out.matrix[0]:
    res = res + v

(1, 4)


In [9]:
type(out)

__main__.Matrix

In [11]:
for i in range(out.rows):
    for j in range(out.cols):
        out.matrix[i][j] = out.matrix[i][j].sigmoid()

TypeError: 'Weight' object is not iterable

In [ ]:
fc1 = FullyConnectedLayer(784, 1024)
fc2 = FullyConnectedLayer(1024, 10)


In [ ]:
from graphviz import Digraph

def trace(root):
  # builds a set of all nodes and edges in a graph
  nodes, edges = set(), set()
  def build(v):
    if v not in nodes:
      nodes.add(v)
      for child in v._prev:
        edges.add((child, v))
        build(child)
  build(root)
  return nodes, edges

def draw_dot(root):
  dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right
  
  nodes, edges = trace(root)
  for n in nodes:
    uid = str(id(n))
    # for any value in the graph, create a rectangular ('record') node for it
    dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f }" % (n.label, n.value, n.grad), shape='record')
    if n._op:
      # if this value is a result of some operation, create an op node for it
      dot.node(name = uid + n._op, label = n._op)
      # and connect this node to it
      dot.edge(uid + n._op, uid)

  for n1, n2 in edges:
    # connect n1 to the op node of n2
    dot.edge(str(id(n1)), str(id(n2)) + n2._op)

  return dot

In [ ]:
draw_dot(res)